# RoadVision AI · Colab training and evaluation

This notebook has not been executed. Select Runtime > Change runtime type > T4 GPU. Training consumes GPU time and disk. Start with the official India subset; do not describe it as the full RDD2022 dataset. Read the official dataset and Ultralytics terms before downloading. No credentials belong in this notebook.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before training.'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import datetime
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d-%H%M%S')
SAVE = Path('/content/drive/MyDrive/RoadVisionAI') / RUN_ID
SAVE.mkdir(parents=True, exist_ok=False)

In [ ]:
REPO = 'https://github.com/techky-Kabilan/Deep-Learning-Lab.git'
BRANCH = 'roadvision-ai-mini-project'
!git clone --depth 1 --branch $BRANCH $REPO /content/Deep-Learning-Lab
%cd /content/Deep-Learning-Lab/mini-projects/roadvision-ai
!pip -q install -r requirements.txt
!git rev-parse HEAD > /content/roadvision-commit.txt
!pip freeze > /content/roadvision-environment.txt

## Retrieve the official archive
The India archive is about 502 MB compressed according to the official repository; extraction requires additional space. This cell downloads data only when you run it. Source and terms: https://github.com/sekilab/RoadDamageDetector . Confirm archive/hosting terms and attribute Arya et al. Dataset images are described as CC BY-SA 4.0 by the official README. Keep the archive outside Git.

In [ ]:
import urllib.request, zipfile, shutil
URL = 'https://bigdatacup.s3.ap-northeast-1.amazonaws.com/2022/CRDDC2022/RDD2022/Country_Specific_Data_CRDDC2022/RDD2022_India.zip'
# If the official S3 endpoint denies access, obtain data from official Figshare DOI 10.6084/m9.figshare.21431547 and place the India subset archive in Drive.
DRIVE_ARCHIVE = None  # Example: '/content/drive/MyDrive/datasets/RDD2022_India.zip'
ARCHIVE = Path(DRIVE_ARCHIVE) if DRIVE_ARCHIVE else Path('/content/India.zip')
RAW = Path('/content/rdd_raw')
assert shutil.disk_usage('/content').free > 4 * 1024**3, 'Need at least 4 GB free space for the India subset; full archive needs much more.'
if not ARCHIVE.exists():
    try:
        urllib.request.urlretrieve(URL, ARCHIVE)
    except Exception as exc:
        if ARCHIVE.exists(): ARCHIVE.unlink()
        raise RuntimeError('Official S3 download unavailable. Obtain the official dataset from Figshare, put an India subset archive in Drive, set DRIVE_ARCHIVE, and rerun. Do not use an unverified mirror.') from exc
RAW.mkdir(exist_ok=True)
with zipfile.ZipFile(ARCHIVE) as archive:
    assert sum(m.file_size for m in archive.infolist()) < shutil.disk_usage('/content').free, 'Not enough space to extract archive.'
    for member in archive.infolist():
        target = (RAW / member.filename).resolve()
        if not target.is_relative_to(RAW.resolve()):
            raise ValueError('Unsafe archive member')
    archive.extractall(RAW)
print('Annotated XML files:', len(list(RAW.rglob('*.xml'))))

## Prepare labelled partitions
Official challenge test images do not include public annotations. Local test metrics use a held-out labelled portion. Default splitting is by distinct image: correlated adjacent scenes may remain. For stronger evidence supply a JSON route/sequence manifest to prepare(). Review support for all classes before training. The output refuses to overwrite an existing dataset.

In [ ]:
from scripts.prepare_rdd2022 import prepare
import json
PREPARED = Path('/content/rdd_prepared')
prepare(RAW, PREPARED, seed=42)
summary = json.loads((PREPARED / 'split_summary.json').read_text())
print(json.dumps(summary, indent=2))
for split in ('train','val','test'):
    assert all(summary['objects'][split].get(c, 0) > 0 for c in ('D00','D10','D20','D40')), f'Missing class support in {split}'
shutil.copy2(PREPARED / 'split_manifest.json', SAVE / 'split_manifest.json')
shutil.copy2(PREPARED / 'split_summary.json', SAVE / 'split_summary.json')

## Fine-tune pretrained YOLOv8n
These are starting hyperparameters, not validated optimum settings. Reduce batch if GPU memory is insufficient. Do not tune on the held-out test set. Restart the runtime if required by the dependency installer before this cell.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
model.train(data=str(PREPARED / 'data.yaml'), epochs=50, imgsz=640, batch=16, device=0, seed=42, deterministic=True, patience=10, workers=2, project='/content/runs', name='roadvision', exist_ok=False)
BEST = Path(model.trainer.best)
assert BEST.is_file()
shutil.copy2(BEST, SAVE / 'best.pt')
shutil.copytree(model.trainer.save_dir, SAVE / 'training')

## Final held-out evaluation
Save aggregate/per-class metrics, confusion matrices (including background), PR curves and provenance. Precision/recall summary values use the library operating point. The low validation confidence threshold is intentional for AP ranking. Inspect false positives and false negatives; document missing/rare classes and domain shift.

In [ ]:
from scripts.evaluate import evaluate
EVALUATION = evaluate(BEST, PREPARED / 'data.yaml', Path('/content/final-evaluation'), split='test', device='0')
shutil.copytree(EVALUATION, SAVE / 'evaluation')
shutil.copy2('/content/roadvision-commit.txt', SAVE / 'commit.txt')
shutil.copy2('/content/roadvision-environment.txt', SAVE / 'environment.txt')
print((EVALUATION / 'metrics.json').read_text())

In [ ]:
from IPython.display import Image, display
for name in ('confusion_matrix.png','confusion_matrix_normalized.png','BoxPR_curve.png'):
    path = EVALUATION / name
    if path.exists(): display(Image(filename=str(path)))
print('Artifacts saved:', SAVE)

## Bring the model into Streamlit
Download best.pt from the run folder to weights/best.pt. Keep weights and data out of Git. Update MODEL_CARD.md and the report only from the saved evaluation evidence. Capture real inference screenshots after installing the checkpoint. The checked-in screenshots use explicitly labelled synthetic fixtures and do not demonstrate model quality.